# Anomaly Detection and Decomposition Demo

## Imports and Environment Setup

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent
sdk_path = project_root / "src" / "sdk" / "python"
sdk_path = sdk_path.resolve()

sys.path.insert(0, str(sdk_path))

In [8]:
DATA_DIR = Path().cwd() / "data" / "ad_demo"

## Load prepared data

In [15]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("Decomposition and AD Demo")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

df = spark.read.parquet(str(DATA_DIR))
df.head()

Row(EventTime=datetime.datetime(2024, 1, 18, 17, 58), Value=380.38916)

## Decomposition of TS data

In [21]:
from rtdip_sdk.pipelines.decomposition.spark.mstl_decomposition import MSTLDecomposition

periods = ['daily']

mstl = MSTLDecomposition(
        df=df,
        value_column='Value',
        timestamp_column='EventTime',
        periods=periods,
        iterate=2,
        stl_kwargs={'robust': False}
    )

decomp_df = mstl.decompose()

/opt/conda/envs/rtdip-sdk/lib/python3.12/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


## Visualize Decomposition

In [24]:
from rtdip_sdk.pipelines.visualization.plotly.decomposition import DecompositionPlotInteractive

decomp_plot = DecompositionPlotInteractive(
    decomposition_data=decomp_df.toPandas(),
    timestamp_column='EventTime',
    value_column='Value',
)

decomp_plot.plot()

## Anomaly Detection

In [ ]:
from rtdip_sdk.pipelines.anomaly_detection.spark.mad.mad_anomaly_detection import (
    DecompositionMadAnomalyDetection,
    GlobalMadScorer
)

mstl_mad = DecompositionMadAnomalyDetection(
    scorer=GlobalMadScorer(),
    decomposition="mstl",
    period="daily",
    timestamp_column="timestamp",
    value_column="value"
)

anomalies = mstl_mad.detect(df
    .withColumnRenamed("EventTime", "timestamp")
    .withColumnRenamed("Value", "value")
)

In [39]:
from rtdip_sdk.pipelines.visualization.plotly.anomaly_detection import AnomalyDetectionPlotInteractive

# Plot full series with anomalies highlighted
ad_plot = AnomalyDetectionPlotInteractive(
    ts_data=df
    .withColumnRenamed("EventTime", "timestamp")
    .withColumnRenamed("Value", "value"),
    ad_data=anomalies,
    title="MSTL + MAD Anomaly Detection"
)

ad_plot.plot()

In [ ]:
# terminate the Spark session
spark.stop()